# Episode 3 — Automatic Differentiation

**Instructor notebook** · run top-to-bottom before recording.

`jax.grad` differentiates scalar functions; `value_and_grad` returns loss and gradients together for training.

| | |
|---|---|
| **Chapter** | 1.3 · Part I — Pure JAX |
| **Prereq** | Episodes 1–2 |
| **Next** | Episode 4 — control flow with JIT |

**JAX docs:** [Automatic differentiation](https://docs.jax.dev/en/latest/automatic-differentiation.html) · [`jax.grad`](https://docs.jax.dev/en/latest/_autosummary/jax.grad.html) · [`jax.value_and_grad`](https://docs.jax.dev/en/latest/_autosummary/jax.value_and_grad.html) · [Working with pytrees](https://docs.jax.dev/en/latest/pytrees.html)


In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr
from jax import grad

## 1. Taking gradients with `jax.grad`

In JAX, you differentiate a **scalar-valued** function with `jax.grad`. It takes a function and returns a function: if `f` evaluates $f$, then `grad(f)(x)` evaluates $\nabla f(x)$.


In [2]:
grad_tanh = grad(jnp.tanh)
print(grad_tanh(2.0))

0.070650935


Because derivative functions are themselves differentiable, you can stack `grad` for higher-order derivatives:

In [3]:
print(grad(grad(jnp.tanh))(2.0))
print(grad(grad(grad(jnp.tanh)))(2.0))

-0.13621888
0.2526544


For $f(x) = x^3 + 2x^2 - 3x + 1$, the derivatives are $f'(x) = 3x^2 + 4x - 3$, $f''(x) = 6x + 4$, $f'''(x) = 6$, $f^{iv}(x) = 0$. Chain `grad` to compute any order:

In [4]:
f = lambda x: x**3 + 2 * x**2 - 3 * x + 1

dfdx = grad(f)
d2fdx = grad(dfdx)
d3fdx = grad(d2fdx)
d4fdx = grad(d3fdx)

print(dfdx(1.0))
print(d2fdx(1.0))
print(d3fdx(1.0))
print(d4fdx(1.0))

4.0
10.0
6.0
0.0


## 2. Gradients in a linear logistic regression

`argnums` selects which positional arguments to differentiate. Default is `0` (first argument).


In [5]:
key = jr.key(0)


def sigmoid(x):
    return 0.5 * (jnp.tanh(x / 2) + 1)


def predict(W, b, inputs):
    return sigmoid(jnp.dot(inputs, W) + b)


inputs = jnp.array(
    [
        [0.52, 1.12, 0.77],
        [0.88, -1.08, 0.15],
        [0.52, 0.06, -1.30],
        [0.74, -2.49, 1.39],
    ]
)
targets = jnp.array([True, True, False, True])


def loss(W, b):
    preds = predict(W, b, inputs)
    label_probs = preds * targets + (1 - preds) * (1 - targets)
    return -jnp.sum(jnp.log(label_probs))


key, W_key, b_key = jr.split(key, 3)
W = jr.normal(W_key, (3,))
b = jr.normal(b_key, ())

In [6]:
W_grad = grad(loss, argnums=0)(W, b) # dLoss/dW
print(f"{W_grad=}")

W_grad = grad(loss)(W, b)
print(f"{W_grad=}")

b_grad = grad(loss, 1)(W, b)
print(f"{b_grad=}")

W_grad, b_grad = grad(loss, (0, 1))(W, b)
print(f"{W_grad=}")
print(f"{b_grad=}")

W_grad=Array([-0.43314588, -0.7354602 , -1.2598921 ], dtype=float32)
W_grad=Array([-0.43314588, -0.7354602 , -1.2598921 ], dtype=float32)
b_grad=Array(-0.6900176, dtype=float32)
W_grad=Array([-0.43314588, -0.7354602 , -1.2598921 ], dtype=float32)
b_grad=Array(-0.6900176, dtype=float32)


With `argnums`, `grad(f, i)` returns a function for $\partial_i f$ — partial derivative with respect to argument $i$.


## 3. Nested dicts and pytrees

JAX's PyTree abstraction lets you differentiate with respect to nested Python containers (dicts, tuples, lists) without extra boilerplate.


In [ ]:
def loss2(params_dict):
    preds = predict(params_dict["W"], params_dict["b"], inputs)
    label_probs = preds * targets + (1 - preds) * (1 - targets)
    return -jnp.sum(jnp.log(label_probs))


print(grad(loss2)({"W": W, "b": b}))

{'W': Array([-0.43314588, -0.7354602 , -1.2598921 ], dtype=float32), 'b': Array(-0.6900176, dtype=float32)}


## 4. `value_and_grad` — loss and gradient in one pass

For training loops, `jax.value_and_grad` computes the function value and gradient together (one forward/backward pass).


In [14]:
loss_value, Wb_grad = jax.value_and_grad(loss, (0, 1))(W, b)
print("loss value", loss_value)
print("loss value", loss(W, b))
print("W grad", Wb_grad[0])
print("b grad", Wb_grad[1])

loss value 2.9729185
loss value 2.9729185
W grad [-0.43314588 -0.7354602  -1.2598921 ]
b grad -0.6900176


## 5. Check against finite differences

Derivatives are easy to sanity-check with finite differences.


In [ ]:
eps = 1e-4

b_grad_numerical = (loss(W, b + eps / 2.0) - loss(W, b - eps / 2.0)) / eps
print("b_grad_numerical", b_grad_numerical)
print("b_grad_autodiff", grad(loss, 1)(W, b))

key, subkey = jr.split(key)
vec = jr.normal(subkey, W.shape)
unitvec = vec / jnp.sqrt(jnp.vdot(vec, vec))
W_grad_numerical = (
    loss(W + eps / 2.0 * unitvec, b) - loss(W - eps / 2.0 * unitvec, b)
) / eps
print("W_dirderiv_numerical", W_grad_numerical)
print("W_dirderiv_autodiff", jnp.vdot(grad(loss)(W, b), unitvec))

b_grad_numerical -0.6866455
b_grad_autodiff -0.6900176
W_dirderiv_numerical 1.2993813
W_dirderiv_autodiff 1.3006742


In [ ]:
from jax.test_util import check_grads

check_grads(loss, (W, b), order=2)

> **Key insight:** Use `value_and_grad` for scalar training losses. `grad` with `argnums` maps cleanly to partial-derivative notation — and PyTrees let you pass params as nested dicts without changing the API.


---
## Exercise

1. Compute `grad(jnp.sin)` at $x = 1.0$ and verify against $(\cos(1), -\sin(1))$ for value and first derivative.
2. Rewrite `loss` to take a single `params` dict; use `value_and_grad` and print both loss and grads.
3. Check `W_grad` with a random direction finite difference (like the demo above) and confirm it matches autodiff.
4. Apply one `grad` to $f(x) = x^4$ and evaluate at $x = 2$ — you should get $4 \cdot 2^3 = 32$.
5. **2-layer MLP training loop** — `init_mlp`, `forward`, and batch `x`/`y` are set up below. Implement `loss_fn` (mean squared error) and fill in the training loop body: `value_and_grad`, an SGD update, and recording the loss each step. Run 20 steps and print the first and last loss.

*(Solution below.)*


In [11]:
print("sin(1):", jnp.sin(1.0), "| cos(1):", jnp.cos(1.0))
print("grad(sin)(1):", grad(jnp.sin)(1.0))

_, params_grad = jax.value_and_grad(
    lambda p: loss(p["W"], p["b"])
)({"W": W, "b": b})
print("params grad:", params_grad)

print("grad(x**4)(2):", grad(lambda x: x**4)(2.0))

sin(1): 0.841471 | cos(1): 0.5403023
grad(sin)(1): 0.5403023
params grad: {'W': Array([-0.43314588, -0.7354602 , -1.2598921 ], dtype=float32), 'b': Array(-0.6900176, dtype=float32)}
grad(x**4)(2): 32.0


In [ ]:
D_IN, D_HIDDEN, D_OUT = 8, 16, 4
B = 32


def init_mlp(key, d_in, d_hidden, d_out):
    key, k0, k1 = jr.split(key, 3)
    return {
        0: {"w": jr.normal(k0, (d_in, d_hidden)) * 0.1, "b": jnp.zeros(d_hidden)},
        1: {"w": jr.normal(k1, (d_hidden, d_out)) * 0.1, "b": jnp.zeros(d_out)},
    }


def forward(params, x):
    # x: (B, D_in) — bias broadcasts over batch
    x = jnp.tanh(x @ params[0]["w"] + params[0]["b"])
    x = x @ params[1]["w"] + params[1]["b"]
    return x


key = jr.key(42)
params = init_mlp(key, D_IN, D_HIDDEN, D_OUT)
key, k_x, k_y = jr.split(key, 3)
x_batch = jr.normal(k_x, (B, D_IN))
y_batch = jr.normal(k_y, (B, D_OUT))


def loss_fn(params, x, y):
    # TODO: mean squared error between forward(params, x) and y
    y_hat = forward(params, x)
    return jnp.mean((y_hat - y) ** 2)


LEARNING_RATE = 0.01
losses = []
for step in range(20):
    loss, grads = jax.value_and_grad(loss_fn)(params, x_batch, y_batch)
    params = jax.tree.map(lambda p, g: p - LEARNING_RATE * g, params, grads)
    # TODO: loss and grads via value_and_grad
    # TODO: SGD update — new params, no in-place mutation
    # TODO: append scalar loss to losses
    pass

print("first loss:", losses[0])
print("last loss:", losses[-1])

IndexError: list index out of range

In [ ]:
def loss_fn(params, x, y):
    return jnp.mean((forward(params, x) - y) ** 2)


LEARNING_RATE = 0.01
losses = []
for step in range(20):
    loss, grads = jax.value_and_grad(loss_fn)(params, x_batch, y_batch)
    params = jax.tree.map(lambda p, g: p - LEARNING_RATE * g, params, grads)
    losses.append(float(loss))

print("first loss:", losses[0])
print("last loss:", losses[-1])

**Next:** Episode 4 — control flow with JIT.
